# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import OGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
os.environ["OPENAI_API_KEY"] = ""

MAX_EPISODES = 5

# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=[('high_elf_mage.yml','Joe'), ('halfling_rogue.yml', 'Roger')],
    enemies=[('high_elf_fighter.yml', 'Mike'), ('halfling_rogue.yml','Spencer')],
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agents[character.name] = (OGPT4Interfacer(debug=True, explain=True, name=character.name), gr, character)


/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:717: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


Joe rolled initiative d20(14) + 2 value 16.15
Roger rolled initiative d20(14) + 5 value 19.2
Mike rolled initiative d20(3) + 5 value 8.2
Spencer rolled initiative d20(4) + 5 value 9.2
Joe rolled initiative d20(5) + 2 value 7.15
Roger rolled initiative d20(11) + 5 value 16.2
Mike rolled initiative d20(16) + 5 value 21.2
Spencer rolled initiative d20(19) + 5 value 24.2
Combat begins with 4 players.
Players: <p>Joe (wizard-2) Team a</p>
<p>Roger (rogue-2) Team a</p>
<p>Mike (fighter-2) Team b</p>
<p>Spencer (rogue-2) Team b</p>
======== Spencer starts their turn. ========
======== Spencer starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [2]:
agents

{'Spencer': (<samples.llm_interface.OGPT4Interfacer at 0x7f94254e4440>,
  'b',
  Spencer),
 'Mike': (<samples.llm_interface.OGPT4Interfacer at 0x7f942536e990>,
  'b',
  Mike),
 'Roger': (<samples.llm_interface.OGPT4Interfacer at 0x7f942527f250>,
  'a',
  Roger),
 'Joe': (<samples.llm_interface.OGPT4Interfacer at 0x7f9425363820>, 'a', Joe)}

In [3]:
def update_all_agents(agents_name, sender, content):
    for name in agents_name:
        agents[name][0].register_conversation(sender, content)

def initiate_conversation(agents_name):
    for name in agents_name:
        agents[name][0].initiate_conversation(sender, content)

def close_conversation(agents_name):
    for name in agents_name:
        agents[name][0].close_conversation(self)

def run_conversation(sender, content):
    sender_gr = agents[sender][1]
    agent_in_the_conv = []
    for name, (_, gr, _) in agents.items():
        if sender_gr == gr and name != sender:
            agent_in_the_conv.append(name)
    agent_in_the_conv.append(sender)
    initiate_conversation(agent_in_the_conv)
    update_all_agents(agent_in_the_conv, sender, content)
    conv_alive = True
    conv_step = 0
    while conv_alive:
        conv_step += 1
        conv_alive = False
        for name in agent_in_the_conv:
            action, content = agents[name][0].select_action_for_state(observation, info, is_conversation=True)
            if action == -2:
                update_all_agents(agent_in_the_conv, name, content)
                conv_alive = True
            elif action != -3:
                raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
    close_conversation(agent_in_the_conv)
    return conv_step


In [4]:
env.env.env.players

[('a', 'H', Joe, [4, 2]),
 ('a', 'H', Roger, [3, 6]),
 ('b', 'E', Mike, [10, 4]),
 ('b', 'E', Spencer, [5, 0])]

In [5]:
# Select an action based on the initial state
current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]

p_observation = env.env.env.generate_observation(current_character)
p_available_moves = compute_available_moves(envi.session, envi.map, current_character, envi.battle, envi.weapon_mappings, envi.spell_mappings)
p_info = envi._info(p_available_moves, current_character)

action = current_agent.select_action_for_state(p_observation, info, env.env.env.players)
print(f"Selected action: {action}")
# terminal = False
# episode = 0
# while not terminal and episode < MAX_EPISODES:
#     episode += 1
#     observation, reward, terminal, truncated, info = env.step(action)
#     if not terminal and not truncated:
#         print(env.render())
#         current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]
#         action = current_agent.select_action_for_state(observation, info)
#         print(f"Selected action: {action}")

#     if terminal or truncated:
#         print(f"Reward: {reward}")
#         break
# action

prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by P (a level 2 rogue).Your health is at [100.]% specifically 16/16 
Your current conditions are:

You have as enemies :
 - Joe denoted by J (a level 2 wizard).
    Their health is currently at 100.0%.
    Their current conditions are: 
 - Roger denoted by R (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: 
You must defeat all of them in order to win.

You are helped in that regard by your allies :
 - Mike denoted by M (a level 2 fighter).
    Their health is currently at 100.0%.
    Their current conditions are: You have the following available actions and movement available:

Available movement: [25]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
____________
____

/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:717: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


KeyError: 'action'

In [7]:
info

{'available_moves': [(0, (0, 0), (-3, 4), 2, 1),
  (0, (0, 0), (-11, 2), 2, 1),
  (0, (0, 0), (-3, 4), 2, 1),
  (0, (0, 0), (-11, 2), 2, 1),
  (0, (0, 0), (-3, 4), 18, 1),
  (0, (0, 0), (-11, 2), 18, 1),
  (15, (-1, -1), (0, 0), 0, 0),
  (4, (-1, -1), (0, 0), 0, 0),
  (5, (-1, -1), (0, 0), 0, 0),
  (2, (-1, -1), (0, 0), 0, 0),
  (11, (-1, -1), (0, 0), 0, 0),
  (3, (-1, -1), (0, 0), 0, 0),
  (1, (-1, -1), (0, 0), 0, 0),
  (1, (-1, 0), (0, 0), 0, 0),
  (1, (-1, 1), (0, 0), 0, 0),
  (1, (0, -1), (0, 0), 0, 0),
  (1, (0, 1), (0, 0), 0, 0),
  (1, (1, -1), (0, 0), 0, 0),
  (1, (1, 0), (0, 0), 0, 0),
  (1, (1, 1), (0, 0), 0, 0),
  (10, (-1, -1), (0, 0), 0, 0),
  (14, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (-1, (0, 0), (0, 0), 0, 0)],
 'current_index': 0,
 'group': 'a',
 'round': 0,
 'health': 16,
 'max_health': 16,
 'weapon_mappings': {'unarmed': 0,
  'battleaxe': 1,
  'dagger': 2,
  'quarterstaff': 3,
  'sling': 4,
  'dart': 5,
  'greatclub

In [25]:
observation.keys()

dict_keys(['map', 'turn_info', 'conditions', 'health_pct', 'player_equipped', 'health_enemy', 'enemy_conditions', 'enemy_reactions', 'player_ac', 'enemy_ac', 'ability_info', 'player_type', 'enemy_type', 'spell_slots', 'movement', 'is_reaction'])

In [8]:
observation["health_enemy"]

array([1.])